In [1]:
!python -m pip install -r requirements.txt

In [3]:
import re #regex
import pandas as pd #pandas
import random #to shuffle the datasets
from sklearn.feature_extraction.text import CountVectorizer #changes the text into numbers
from sklearn.model_selection import train_test_split #splits the data into training and testing data
from sklearn.linear_model import LogisticRegression #uses multiclass logistic regression
from sklearn.metrics import classification_report, accuracy_score #imports evaluator for logistic regression
import pickle #to save the model

# get access to the data
df = pd.read_csv("Option1-TrainingDataset.csv")

#Preprocessing
def clean_text(text):
    if not isinstance(text, str): #if not a string replace with empty string
        text = ''

    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Remove links
    text = re.sub(r'#', '', text) #Remove hashtags but keeps word after it
    text = re.sub(r'[^\w\s]', '', text) # Removes punctuation
    return text



# add variables for columns to store
texts = df['text'].values
labels = df['label'].values

texts = pd.Series(texts).apply(clean_text).values



#emotion labels to make it clearer in the report than using the numbers
emotion_labels = ['Anger (0)', 'Fear (1)', 'Joy (2)', 'Love (3)', 'Sadness (4)', 'Surprise (5)']

# change the text into numbers
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts.astype('U'))

# split training data 80% and testing data 20%
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42,stratify=labels)


#create multiclass logistic regression model
model_lr = LogisticRegression(
    solver='saga',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

print("Starting training")
model_lr.fit(X_train, y_train)
print("Training complete.")
print("\n")

#Predict what emotion it should be
y_pred = model_lr.predict(X_test)

#Creates report showing precision, recall, and f1-score for the emotions
print("Classification Report (for the 6 emotions):")
print(classification_report(y_test, y_pred,target_names=emotion_labels))
print("\n")


#get the words from count vectorizer
feature_names = vectorizer.get_feature_names_out()

coefficients = model_lr.coef_

print("\n")
print("Top 3 most influential words per emotion")

# loop through each emotion
for i, emotion in enumerate(emotion_labels):
    #get the weights and sorts it into order, takes the top 3
    top_3_indices = coefficients[i].argsort()[-3:][::-1]

    #gets the words
    top_3_words = [feature_names[j] for j in top_3_indices]

    #get the weights
    top_3_values = [coefficients[i][j] for j in top_3_indices]

    #prints the top words for each emotion
    print(f"\n{emotion}:")
    for word, value in zip(top_3_words, top_3_values):
        print(f"  - '{word}' (Weight: {value:.4f})")

#save the model and vectorizer
with open('logistic_model.pkl', 'wb') as file:
    pickle.dump(model_lr, file)

with open('count_vectorizer.pkl', 'wb') as file:
    pickle.dump(vectorizer, file)

print("Saved  in 'count_vectorizer' and 'logistic_model' directory")

Starting training


c:\Users\olive\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Training complete.


Classification Report (for the 6 emotions):
              precision    recall  f1-score   support

   Anger (0)       0.92      0.93      0.92       933
    Fear (1)       0.88      0.93      0.91      1072
     Joy (2)       0.82      0.74      0.78       261
    Love (3)       0.90      0.87      0.88       432
 Sadness (4)       0.84      0.83      0.84       387
Surprise (5)       0.82      0.73      0.77       115

    accuracy                           0.89      3200
   macro avg       0.87      0.84      0.85      3200
weighted avg       0.89      0.89      0.89      3200





Top 3 most influential words per emotion

Anger (0):
  - 'troubled' (Weight: 3.0291)
  - 'disturbed' (Weight: 2.9494)
  - 'lethargic' (Weight: 2.9372)

Fear (1):
  - 'innocent' (Weight: 2.6700)
  - 'resolved' (Weight: 2.5936)
  - 'satisfied' (Weight: 2.5467)

Joy (2):
  - 'sympathetic' (Weight: 3.9538)
  - 'nostalgic' (Weight: 3.9404)
  - 'naughty' (Weight: 3.8590)

Love (3):
  - 'gree